In [8]:
!cp -r /kaggle/input/sar-project/SAR-Denoising-Project /kaggle/working/

In [9]:
!cp -r /kaggle/input/sar-project/SAR-Denoising-Project/Models /kaggle/working/
!cp -r /kaggle/input/sar-project/SAR-Denoising-Project/ETL /kaggle/working/


In [ ]:
# Display dncnn.py with line numbers to edit
with open("/kaggle/working/Models/DnCNN/dncnn.py", "r") as f:
    content = f.read()

print(content)


# dncnn.py
# Architecture DnCNN + chargement des poids préentraînés + outils de freeze

import torch
import torch.nn as nn
from collections import OrderedDict


class DnCNN(nn.Module):
    """
    Implémentation PyTorch du DnCNN (17 couches par défaut).
    Le réseau apprend à prédire le bruit → clean = input - noise.
    """

    def __init__(self, depth=17, n_channels=64, image_channels=1, use_bnorm=True):
        super(DnCNN, self).__init__()

        layers = []

        # 1️⃣ Première couche (Conv + ReLU)
        layers.append(nn.Conv2d(image_channels, n_channels, kernel_size=3, padding=1, bias=True))
        layers.append(nn.ReLU(inplace=True))

        # 2️⃣ Couches 2..(depth-1) : Conv + BN + ReLU
        for _ in range(depth - 2):
            layers.append(nn.Conv2d(n_channels, n_channels, kernel_size=3, padding=1, bias=False))
            if use_bnorm:
                layers.append(nn.BatchNorm2d(n_channels))
            layers.append(nn.ReLU(inplace=True))

        # 3️⃣ Dern

In [11]:
import os
print(os.path.exists("/kaggle/working/Models/DnCNN/pretrained/net.pth"))


True


In [12]:
code = """# dncnn.py
# Architecture DnCNN + pretrained weight loader + freeze utilities

import os
import torch
import torch.nn as nn
from collections import OrderedDict


class DnCNN(nn.Module):
    # PyTorch implementation of DnCNN (17 layers).
    # The network predicts noise: clean = noisy - noise.

    def __init__(self, depth=17, n_channels=64, image_channels=1, use_bnorm=True):
        super().__init__()

        layers = []

        # First layer — Conv + ReLU
        layers.append(nn.Conv2d(image_channels, n_channels, 3, padding=1, bias=False)) 
        layers.append(nn.ReLU(inplace=True))

        # Middle layers — Conv + BN + ReLU
        for _ in range(depth - 2):
            layers.append(nn.Conv2d(n_channels, n_channels, 3, padding=1, bias=False))
            if use_bnorm:
                layers.append(nn.BatchNorm2d(n_channels))
            layers.append(nn.ReLU(inplace=True))

        # Final layer — Conv
        layers.append(nn.Conv2d(n_channels, image_channels, 3, padding=1, bias=False))

        self.dncnn = nn.Sequential(*layers)

    def forward(self, x):
        noise = self.dncnn(x)
        return x - noise


# ---------------------------------------------------------
# LOAD PRETRAINED WEIGHTS
# ---------------------------------------------------------

def load_pretrained_dncnn_s(model):
    weight_path = (
        "/kaggle/working/SAR-Denoising-Project/"
        "Models/DnCNN/pretrained/net.pth"
    )

    if not os.path.exists(weight_path):
        raise FileNotFoundError(
            f"Pretrained weights not found:\\n{weight_path}"
        )

    state_dict = torch.load(weight_path, map_location="cpu")

    clean_state_dict = {
        k.replace("module.", ""): v for k, v in state_dict.items()
    }

    model.load_state_dict(clean_state_dict)
    print("[OK] Pretrained DnCNN-S weights loaded.")

    return model


# ---------------------------------------------------------
# FREEZE LAYERS
# ---------------------------------------------------------

def freeze_first_layers(model, num_layers_to_freeze):
    count = 0
    for layer in model.dncnn.children():
        if count < num_layers_to_freeze:
            for param in layer.parameters():
                param.requires_grad = False
        count += 1
    print(f"[INFO] {num_layers_to_freeze} first layers frozen.")


def count_trainable(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"[INFO] Total params     : {total:,}")
    print(f"[INFO] Trainable params : {trainable:,}")
    return total, trainable


# ---------------------------------------------------------
# QUICK SELF-TEST
# ---------------------------------------------------------

if __name__ == "__main__":
    model = DnCNN()
    x = torch.randn(1, 1, 64, 64)
    y = model(x)
    print("Output shape:", y.shape)
"""

In [ ]:
with open("/kaggle/working/SAR-Denoising-Project/Models/DnCNN/dncnn.py", "w") as f:
    f.write(code)

print("dncnn.py updated correctly ✔️")

dncnn.py updated correctly ✔️


In [ ]:
import os
os._exit(0)


In [1]:
import sys, os, torch

# Add paths
sys.path.append("/kaggle/working/SAR-Denoising-Project/Models/DnCNN")
sys.path.append("/kaggle/working/SAR-Denoising-Project/ETL/src")

# Import modules
from dncnn import DnCNN, load_pretrained_dncnn_s
from dataset_log_multiL import SARDenoiseMultiL_Log

# Load dataset
print("Loading dataset...")
ds = SARDenoiseMultiL_Log("/kaggle/working/SAR-Denoising-Project/ETL/data/pickles/patches_multiL_log.pkl")
print("Dataset OK:", len(ds))

# Get sample
x, y = ds[0]
x = x.unsqueeze(0)

# Load pretrained model
print("Loading model...")
model = DnCNN(depth=17, n_channels=64, image_channels=1)
model = load_pretrained_dncnn_s(model)

# Forward pass
print("Running forward test...")
with torch.no_grad():
    out = model(x)

print("Forward OK. Output shape:", out.shape)
print("NO CRASH — Ready for finetuning ✔️")


Loading dataset...
[INFO] Loaded 40144 log-domain pairs from ['L1', 'L2', 'L4', 'L8']
Dataset OK: 40144
Loading model...
[OK] Pretrained DnCNN-S weights loaded.
Running forward test...
Forward OK. Output shape: torch.Size([1, 1, 64, 64])
NO CRASH — Ready for finetuning ✔️


In [2]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0))



CUDA available: True
GPU name: Tesla T4


In [3]:
import os
os.chdir("/kaggle/working/SAR-Denoising-Project/ETL/src")
print("Current working dir:", os.getcwd())


Current working dir: /kaggle/working/SAR-Denoising-Project/ETL/src


In [4]:
import sys
sys.path.append("/kaggle/working/SAR-Denoising-Project/Models/DnCNN")
print("DnCNN path added.")


DnCNN path added.


In [5]:
import dncnn
print("Loaded", dncnn)


Loaded <module 'dncnn' from '/kaggle/working/SAR-Denoising-Project/Models/DnCNN/dncnn.py'>


In [ ]:
# Display dncnn.py with line numbers to edit later
with open("/kaggle/working/ETL/src/train_finetune.py", "r") as f:
    content = f.read()

print(content)


# train_finetune.py
# Fine-tuning du modèle DnCNN-S (blind denoiser) sur vos patches log-domain

import os, sys
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.optim import Adam
from torch.optim.lr_scheduler import StepLR

# Pour importer le modèle depuis /Models
sys.path.append(os.path.abspath("../../Models"))

from dncnn import DnCNN, load_pretrained_dncnn_s, freeze_first_layers
from dataset_log_multiL import SARDenoiseMultiL_Log


# ----------------------------------------------------------------------
# 1. CONFIGURATION
# ----------------------------------------------------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

CONFIG = {
    "batch_size": 16,
    "epochs": 5,                       # safe pour CPU
    "learning_rate": 1e-4,
    "num_workers": 0,                  # mettre >0 si GPU
    "freeze_layers": 12,               # on fine-tune seulement les couches finales
    "checkpoint_path": "../checkpoints/dncnn_

In [ ]:
code = """# train_finetune.py
# Fine-tuning du modèle DnCNN-S (blind denoiser) sur vos patches log-domain

import sys
sys.path.insert(0, "/kaggle/working/SAR-Denoising-Project")
sys.path.append("/kaggle/working/SAR-Denoising-Project")

from Models.DnCNN.dncnn import DnCNN, load_pretrained_dncnn_s, freeze_first_layers


import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.optim import Adam
from torch.optim.lr_scheduler import StepLR


from dataset_log_multiL import SARDenoiseMultiL_Log


# ----------------------------------------------------------------------
# 1. CONFIGURATION
# ----------------------------------------------------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

CONFIG = {
    "batch_size": 16,
    "epochs": 5,                       # initialement pour le test à modifer dans la suite du notebook
    "learning_rate": 1e-4,             # modifié après dans le notebook
    "num_workers": 0,                  # modifié après dans le notebook
    "freeze_layers": 12,               # on fine-tune seulement les couches finales
    "checkpoint_path": "../checkpoints/dncnn_finetuned.pth",
    "pkl_path": "../data/pickles/patches_multiL_log.pkl"
}


# ----------------------------------------------------------------------
# 2. TRAINING FUNCTION
# ----------------------------------------------------------------------
def train_one_epoch(model, dataloader, optimizer, criterion, epoch):
    model.train()
    running_loss = 0.0

    for i, (noisy, clean) in enumerate(dataloader):
        noisy = noisy.to(DEVICE)
        clean = clean.to(DEVICE)

        optimizer.zero_grad()
        output = model(noisy)
        loss = criterion(output, clean)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        if i % 100 == 0:
            print(f"  Batch {i}/{len(dataloader)} — Loss: {loss.item():.6f}")

    avg_loss = running_loss / len(dataloader)
    print(f"[EPOCH {epoch}] Avg Loss = {avg_loss:.6f}")
    return avg_loss


# ----------------------------------------------------------------------
# 3. MAIN TRAINING LOOP
# ----------------------------------------------------------------------
def finetune_dncnn():

    print("\n==============================")
    print("   FINE-TUNING DnCNN-S")
    print("==============================\n")

    # --- Load dataset ---
    print("[INFO] Loading dataset...")
    dataset = SARDenoiseMultiL_Log(CONFIG["pkl_path"])
    dataloader = DataLoader(
        dataset,
        batch_size=CONFIG["batch_size"],
        shuffle=True,
        num_workers=CONFIG["num_workers"]
    )

    # --- Load pretrained model ---
    model = DnCNN(depth=17, n_channels=64, image_channels=1).to(DEVICE)
    model = load_pretrained_dncnn_s(model)

    # --- Freeze early layers, fine-tune last ones ---
    freeze_first_layers(model, CONFIG["freeze_layers"])

    # Loss + optimizer
    criterion = nn.MSELoss()
    optimizer = Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=CONFIG["learning_rate"]
    )

    scheduler = StepLR(optimizer, step_size=2, gamma=0.5)

    # --- Create checkpoint folder if not exists ---
    os.makedirs(os.path.dirname(CONFIG["checkpoint_path"]), exist_ok=True)

    # --- Training loop ---
    best_loss = float("inf")

    for epoch in range(1, CONFIG["epochs"] + 1):
        avg_loss = train_one_epoch(
            model,
            dataloader,
            optimizer,
            criterion,
            epoch
        )

        scheduler.step()

        # Save best model
        if avg_loss < best_loss:
            best_loss = avg_loss
            torch.save(model.state_dict(), CONFIG["checkpoint_path"])
            print(f"[SAVE] New best model saved at epoch {epoch}\n")

    print("\n==============================")
    print("   TRAINING FINISHED 🎉")
    print("==============================")
    print(f"Best loss: {best_loss:.6f}")
    print(f"Checkpoint saved to: {CONFIG['checkpoint_path']}")


# ----------------------------------------------------------------------
# ENTRY POINT
# ----------------------------------------------------------------------
if __name__ == "__main__":
    finetune_dncnn()
"""

In [9]:
with open("/kaggle/working/ETL/src/train_finetune.py", "w") as f:
    f.write(code)

print("train_finetune.py updated correctly ✔️")

train_finetune.py updated correctly ✔️


In [ ]:
import os
os._exit(0)

In [1]:
!python /kaggle/working/SAR-Denoising-Project/ETL/src/train_finetune.py


Traceback (most recent call last):
  File "/kaggle/working/SAR-Denoising-Project/ETL/src/train_finetune.py", line 14, in <module>
    from dncnn import DnCNN, load_pretrained_dncnn_s, freeze_first_layers
ModuleNotFoundError: No module named 'dncnn'


In [2]:
!grep -R "from dncnn import" -n SAR-Denoising-Project


SAR-Denoising-Project/ETL/src/train_finetune.py:14:from dncnn import DnCNN, load_pretrained_dncnn_s, freeze_first_layers


In [3]:
!sed -n '1,30p' SAR-Denoising-Project/ETL/src/train_finetune.py


# train_finetune.py
# Fine-tuning du modèle DnCNN-S (blind denoiser) sur vos patches log-domain

import os, sys
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.optim import Adam
from torch.optim.lr_scheduler import StepLR

# Pour importer le modèle depuis /Models
sys.path.append(os.path.abspath("../../Models"))

from dncnn import DnCNN, load_pretrained_dncnn_s, freeze_first_layers
from dataset_log_multiL import SARDenoiseMultiL_Log


# ----------------------------------------------------------------------
# 1. CONFIGURATION
# ----------------------------------------------------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

CONFIG = {
    "batch_size": 16,
    "epochs": 5,                       # safe pour CPU
    "learning_rate": 1e-4,
    "num_workers": 0,                  # mettre >0 si GPU
    "freeze_layers": 12,               # on fine-tune seulement les couches finales
    "checkpoint_path": "../checkpoints/dncnn_

In [4]:
!sed -i 's/from dncnn import DnCNN, load_pretrained_dncnn_s, freeze_first_layers/from Models.DnCNN.dncnn import DnCNN, load_pretrained_dncnn_s, freeze_first_layers/' SAR-Denoising-Project/ETL/src/train_finetune.py


In [5]:
!sed -n '1,30p' SAR-Denoising-Project/ETL/src/train_finetune.py


# train_finetune.py
# Fine-tuning du modèle DnCNN-S (blind denoiser) sur vos patches log-domain

import os, sys
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.optim import Adam
from torch.optim.lr_scheduler import StepLR

# Pour importer le modèle depuis /Models
sys.path.append(os.path.abspath("../../Models"))

from Models.DnCNN.dncnn import DnCNN, load_pretrained_dncnn_s, freeze_first_layers
from dataset_log_multiL import SARDenoiseMultiL_Log


# ----------------------------------------------------------------------
# 1. CONFIGURATION
# ----------------------------------------------------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

CONFIG = {
    "batch_size": 16,
    "epochs": 5,                       # safe pour CPU
    "learning_rate": 1e-4,
    "num_workers": 0,                  # mettre >0 si GPU
    "freeze_layers": 12,               # on fine-tune seulement les couches finales
    "checkpoint_path": "../check

In [1]:
!python SAR-Denoising-Project/ETL/src/train_finetune.py



   FINE-TUNING DnCNN-S

[INFO] Loading dataset...
Traceback (most recent call last):
  File "/kaggle/working/SAR-Denoising-Project/ETL/src/train_finetune.py", line 134, in <module>
    finetune_dncnn()
  File "/kaggle/working/SAR-Denoising-Project/ETL/src/train_finetune.py", line 76, in finetune_dncnn
    dataset = SARDenoiseMultiL_Log(CONFIG["pkl_path"])
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/kaggle/working/SAR-Denoising-Project/ETL/src/dataset_log_multiL.py", line 13, in __init__
    with open(pkl_path, "rb") as f:
         ^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: '../data/pickles/patches_multiL_log.pkl'


In [6]:
!sed -i '1i import sys\nsys.path.insert(0, "/kaggle/working/SAR-Denoising-Project")\n' SAR-Denoising-Project/ETL/src/train_finetune.py


In [7]:
!sed -n '1,25p' SAR-Denoising-Project/ETL/src/train_finetune.py


import sys
sys.path.insert(0, "/kaggle/working/SAR-Denoising-Project")

# train_finetune.py
# Fine-tuning du modèle DnCNN-S (blind denoiser) sur vos patches log-domain

import os, sys
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.optim import Adam
from torch.optim.lr_scheduler import StepLR

# Pour importer le modèle depuis /Models
sys.path.append(os.path.abspath("../../Models"))

from Models.DnCNN.dncnn import DnCNN, load_pretrained_dncnn_s, freeze_first_layers
from dataset_log_multiL import SARDenoiseMultiL_Log


# ----------------------------------------------------------------------
# 1. CONFIGURATION
# ----------------------------------------------------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"



In [2]:
!ls /kaggle/working/SAR-Denoising-Project/data/pickles


ls: cannot access '/kaggle/working/SAR-Denoising-Project/data/pickles': No such file or directory


In [8]:
!sed -i 's|pkl_path":.*|pkl_path": "/kaggle/working/SAR-Denoising-Project/ETL/data/pickles/patches_multiL_log.pkl"|g' SAR-Denoising-Project/ETL/src/train_finetune.py


In [9]:
!grep -n "pkl_path" SAR-Denoising-Project/ETL/src/train_finetune.py


33:    "pkl_path": "/kaggle/working/SAR-Denoising-Project/ETL/data/pickles/patches_multiL_log.pkl"
76:    dataset = SARDenoiseMultiL_Log(CONFIG["pkl_path"])


In [10]:
!sed -i \
-e 's/"epochs": *[0-9]\+/"epochs": 25/' \
-e 's/"freeze_layers": *[0-9]\+/"freeze_layers": 8/' \
-e 's/"learning_rate": *[0-9.e-]\+/"learning_rate": 5e-5/' \
SAR-Denoising-Project/ETL/src/train_finetune.py


In [11]:
!grep -n '"epochs"\|"freeze_layers"\|"learning_rate"' SAR-Denoising-Project/ETL/src/train_finetune.py


28:    "epochs": 25,                       # safe pour CPU
29:    "learning_rate": 5e-5,
31:    "freeze_layers": 8,               # on fine-tune seulement les couches finales
89:    freeze_first_layers(model, CONFIG["freeze_layers"])
95:        lr=CONFIG["learning_rate"]
106:    for epoch in range(1, CONFIG["epochs"] + 1):


in the previous cells we forced some path modifications in kaggle to fix paths issues, now we can launch the train_finetune.py

In [12]:
!python SAR-Denoising-Project/ETL/src/train_finetune.py



   FINE-TUNING DnCNN-S

[INFO] Loading dataset...
[INFO] Loaded 40144 log-domain pairs from ['L1', 'L2', 'L4', 'L8']
[OK] Pretrained DnCNN-S weights loaded.
[INFO] 8 first layers frozen.
  Batch 0/2509 — Loss: 32.589924
  Batch 100/2509 — Loss: 33.073257
  Batch 200/2509 — Loss: 36.975716
  Batch 300/2509 — Loss: 494.898773
  Batch 400/2509 — Loss: 40.531055
  Batch 500/2509 — Loss: 55.318951
  Batch 600/2509 — Loss: 21.842743
  Batch 700/2509 — Loss: 35.308315
  Batch 800/2509 — Loss: 43.995110
  Batch 900/2509 — Loss: 16.934032
  Batch 1000/2509 — Loss: 1016.879944
  Batch 1100/2509 — Loss: 44.545296
  Batch 1200/2509 — Loss: 48.712921
  Batch 1300/2509 — Loss: 16.493542
  Batch 1400/2509 — Loss: 25.835110
  Batch 1500/2509 — Loss: 41.474915
  Batch 1600/2509 — Loss: 311.005127
  Batch 1700/2509 — Loss: 68.206482
  Batch 1800/2509 — Loss: 18.304453
  Batch 1900/2509 — Loss: 3.866994
  Batch 2000/2509 — Loss: 25.781425
  Batch 2100/2509 — Loss: 22.175255
  Batch 2200/2509 — Loss: 4.9

In [43]:
import os
import torch
import numpy as np
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
from torchvision import transforms
from PIL import Image

# -----------------------------
# CONFIG
# -----------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

PROJECT_ROOT = "/kaggle/working/SAR-Denoising-Project"
CHECKPOINT = PROJECT_ROOT + "/ETL/checkpoints/dncnn_finetuned.pth"
DATA_ROOT = PROJECT_ROOT + "/ETL/data/processed"

EPS = 1e-6

# -----------------------------
# INVERSE LOG TRANSFORM (CORRECT)
# -----------------------------
def inverse_log_transform(x, eps=1e-6):
    return np.exp(x) - eps

# -----------------------------
# LOAD MODEL
# -----------------------------
from Models.DnCNN.dncnn import DnCNN

model = DnCNN(depth=17, n_channels=64, image_channels=1)
model.load_state_dict(torch.load(CHECKPOINT, map_location=DEVICE))
model = model.to(DEVICE)
model.eval()

# -----------------------------
# IMAGE TRANSFORM
# -----------------------------
to_tensor = transforms.ToTensor()

# -----------------------------
# METRICS
# -----------------------------
results = {}

for L in ["L1", "L2", "L4", "L8"]:

    folder = os.path.join(DATA_ROOT, L, "bsd68")
    assert os.path.exists(folder), f"Missing folder: {folder}"

    psnr_list = []
    ssim_list = []

    # list all *_noisy.jpg files
    noisy_files = sorted([f for f in os.listdir(folder) if f.endswith("_noisy.jpg")])

    for nf in noisy_files:
        cf = nf.replace("_noisy.jpg", "_clean.jpg")

        noisy_img = Image.open(os.path.join(folder, nf)).convert("L")
        clean_img = Image.open(os.path.join(folder, cf)).convert("L")

        noisy = to_tensor(noisy_img).unsqueeze(0).to(DEVICE)
        clean = to_tensor(clean_img).squeeze().numpy()

        with torch.no_grad():
            denoised = model(noisy).squeeze().cpu().numpy()

        # inverse log
        clean_lin = inverse_log_transform(clean, EPS)
        den_lin   = inverse_log_transform(denoised, EPS)
        max_val = np.percentile(clean_lin, 99.9)

        clean_lin = np.clip(clean_lin, 0, max_val)
        den_lin   = np.clip(den_lin, 0, max_val)

        data_range = clean_lin.max()

        psnr = peak_signal_noise_ratio(clean_lin, den_lin, data_range=data_range)
        ssim = structural_similarity(clean_lin, den_lin, data_range=data_range)

        psnr_list.append(psnr)
        ssim_list.append(ssim)

    results[L] = {
        "PSNR_mean": float(np.mean(psnr_list)),
        "SSIM_mean": float(np.mean(ssim_list))
    }

# -----------------------------
# PRINT RESULTS
# -----------------------------
print("\nFINAL RESULTS (FULL IMAGES, AFTER INVERSE LOG):\n")
for L, v in results.items():
    print(f"{L}:  PSNR = {v['PSNR_mean']:.2f} dB | SSIM = {v['SSIM_mean']:.4f}")



FINAL RESULTS (FULL IMAGES, AFTER INVERSE LOG):

L1:  PSNR = 11.26 dB | SSIM = 0.4850
L2:  PSNR = 10.89 dB | SSIM = 0.3788
L4:  PSNR = 11.00 dB | SSIM = 0.4379
L8:  PSNR = 11.26 dB | SSIM = 0.4849
